# Evaluating a model on MedCalc-Bench

`mk.evaluate_model(model)` evaluates a model at two levels:

- **general**: the original task (end to end) on the whole benchmark: 55 calculators, from lab formulas to
  rule-based scores and dates;
- **step by step**: on the 33 equation-based calculators, each step of the calculation is measured on its own:
  **E** finding the values in the note, **K** knowing the formula, **S** substituting the values, **A** computing.
  Next to the end-to-end result on the same examples, it shows where the chain breaks.

This notebook runs it on Qwen2.5-3B-Instruct with a GPU. To evaluate another model, change `MODEL_ID`, or replace
`model` by any function `generate(messages, max_new_tokens) -> text` (an API client, vLLM...). Set
`PER_CALCULATOR = None` for the full test set (1,100 examples, 3,740 prompts).

In [1]:
import medcalc_kit as mk
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 160)
print("medcalc-kit", mk.__version__)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
PER_CALCULATOR = 5          # examples per calculator (None: the whole test split)
BATCH_SIZE = 16
OUTPUT_DIR = "outputs/eval-qwen2.5-3b"

medcalc-kit 0.5.0


## The model

Any chat model works. Here the Hugging Face backend loads it on the GPU with the most free memory and decodes
greedily.

In [2]:
import torch
import transformers
from medcalc_kit.generation.hf import HFBackend

transformers.utils.logging.disable_progress_bar()
free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
device = f"cuda:{max(range(len(free)), key=free.__getitem__)}"
model = HFBackend(MODEL_ID, device=device)
print(model.model_id, "on", device)

/home/merveil/miniconda3/envs/medcalc-kit/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen/Qwen2.5-3B-Instruct on cuda:0


## The evaluation

One call: it loads the benchmark, builds the prompts, generates (each answer is saved as it comes, so the same call
resumes an interrupted run), scores, and saves the report in `OUTPUT_DIR`.

In [3]:
import time

t0 = time.time()
report = mk.evaluate_model(model, OUTPUT_DIR, per_calculator=PER_CALCULATOR, batch_size=BATCH_SIZE)
print(f"done in {time.time() - t0:.0f} s, {len(report.items)} prompts\n")
report

done in 8 s, 1055 prompts



MedCalc-Bench evaluation of Qwen/Qwen2.5-3B-Instruct
  general (end to end): 16.7% of 275 examples (55 calculators)
  step by step (165 examples of equation-based calculators):
    E    extraction          86.1%
    U    unit conversion     74.2%
    K    formula knowledge    3.6%
    S    substitution        73.3%
    A    arithmetic          61.2%
    E2E  end to end          20.6%
  consistency: E2E correct for 80% of the examples whose steps all succeed, 19% when a step fails

## General score

End-to-end accuracy per category of calculator. `well_formed` is the share of answers in the expected format; only
those can be correct.

In [4]:
report.general.style.format({"well_formed": "{:.0%}", "accuracy": "{:.1%}"})

,n,well_formed,accuracy
category,,,
all,275,96%,16.7%
date,15,100%,6.7%
diagnosis,15,93%,20.0%
dosage,10,100%,0.0%
lab test,95,94%,14.7%
physical,60,100%,33.3%
risk,60,95%,10.0%
severity,20,95%,10.0%


## Step by step

On the equation-based calculators: the accuracy of each step, and of the end-to-end task on the same examples.
`competence` is the accuracy among well-formed answers, so format failures and reasoning failures are separated.

In [5]:
report.steps.style.format({"well_formed": "{:.0%}", "competence": "{:.1%}", "accuracy": "{:.1%}"})

,n,well_formed,competence,accuracy
step,,,,
E,165,100%,86.1%,86.1%
U,120,100%,74.2%,74.2%
K,165,84%,4.3%,3.6%
S,165,78%,94.5%,73.3%
A,165,79%,77.1%,61.2%
E2E,165,96%,21.4%,20.6%


For each example, the first step that fails (in the order E, K, S, A; `none` if all four succeed), and how often
the end-to-end answer is right in each case.

In [6]:
report.first_failures.style.format({"share": "{:.0%}", "e2e_accuracy": "{:.0%}"})

,examples,share,e2e_accuracy
first_failure,,,
E,23,14%,9%
U,25,15%,20%
K,111,67%,20%
A,1,1%,100%
none,5,3%,80%


In [7]:
report.examples.head(10)

,example_id,calculator_id,category,E,U,K,S,A,E2E,first_failure
0,test:10:162,10,physical,True,True,False,True,True,True,K
1,test:10:169,10,physical,True,True,False,True,True,False,K
2,test:10:171,10,physical,True,True,False,True,False,False,K
3,test:10:179,10,physical,False,True,False,True,False,False,E
4,test:10:180,10,physical,True,True,False,True,False,False,K
5,test:11:182,11,physical,True,NaN,False,False,True,False,K
6,test:11:189,11,physical,True,NaN,False,True,True,True,K
7,test:11:191,11,physical,True,NaN,False,False,True,False,K
8,test:11:199,11,physical,True,NaN,False,False,True,False,K
9,test:11:200,11,physical,True,NaN,False,True,True,False,K


## Per calculator

Where the chain breaks, calculator by calculator. Rule-based and date calculators only have the end-to-end
column.

In [8]:
table = report.by_calculator.set_index(["calculator_id", "calculator_name", "category"])
table.style.format("{:.0%}", na_rep="").background_gradient(cmap="RdYlGn", vmin=0, vmax=1)

,,,E,U,K,S,A,E2E
calculator_id,calculator_name,category,,,,,,
2,Creatinine Clearance (Cockcroft-Gault Equation),lab test,0%,100%,0%,0%,40%,0%
3,CKD-EPI Equations for Glomerular Filtration Rate,lab test,100%,0%,0%,100%,0%,0%
4,CHA2DS2-VASc Score for Atrial Fibrillation Stroke Risk,risk,,,,,,0%
5,Mean Arterial Pressure (MAP),physical,100%,,80%,100%,80%,80%
6,Body Mass Index (BMI),physical,100%,100%,40%,100%,100%,100%
7,Calcium Correction for Hypoalbuminemia,lab test,100%,60%,0%,60%,20%,60%
8,Wells' Criteria for Pulmonary Embolism,risk,,,,,,20%
9,MDRD GFR Equation,lab test,20%,0%,0%,100%,0%,40%
10,Ideal Body Weight,physical,80%,100%,0%,100%,40%,20%


## Errors

Every answer is in `report.scores`, with its text and the scorer's verdict. The most frequent error types per step:

In [9]:
s = report.scores
(s[~s.correct].groupby(["task", "error_type"], dropna=False).size().rename("answers")
 .reset_index().sort_values(["task", "answers"], ascending=[True, False]).groupby("task").head(3))

,task,error_type,answers
0,A,arithmetic_error,30
2,A,format_invalid,27
3,A,truncated,6
5,E,unit_error,9
6,E,wrong_value,8
4,E,missing,6
9,E2E,out_of_bounds,204
11,E2E,wrong_date,10
7,E2E,degenerate,5
17,K,wrong_at_patient_point,131


Answers that are not usable (empty, cut off by the token budget, looping, or not in the expected format), per step:

In [10]:
pd.crosstab(s.task, s.category)

category,degenerate,format_invalid,truncated,well_formed
task,,,,
A,1,27,6,131
E,0,0,0,165
E2E,5,2,4,264
K,0,24,3,138
S,0,37,0,128
U,0,0,0,120


One failure per step, to read the model's answer and the verdict:

In [11]:
for task in ["E", "K", "S", "A", "E2E"]:
    fails = s[(s.task == task) & s.well_formed & ~s.correct]
    if fails.empty:
        continue
    row = fails.iloc[0]
    verdict = {k: v for k, v in row.verdict.items() if k not in ("points", "fields")}
    print(f"=== {task} {row.item_id} ({row.calculator_name}) -> {row.error_type}")
    print(row.text.strip()[-400:])
    print(verdict, "\n")

=== E test:24:1020:E (Steroid Conversion Calculator) -> wrong_value
{"input steroid": {"drug": "null", "value": "null", "unit": "null"}}
{'format_valid': True, 'correct': False, 'field_accuracy': 0.0, 'error_type': 'wrong_value'} 

=== K test:10:162:K (Ideal Body Weight) -> wrong_at_patient_point
{
  "formula": "(max(46.7, round((height - 50) * 1.9)))"
}
{'format_valid': True, 'correct': False, 'error_type': 'wrong_at_patient_point', 'point_pass_rate': 0.0, 'original_point_correct': False, 'n_points': 51} 

=== S test:23:342:S (MELD Na (UNOS/OPTN)) -> structure_or_assignment_error
{"expression": "round(round(0.957 * ln(min(max(1.6, 1), 4)) + 0.378 * ln(max(1.11, 1)) + 1.12 * ln(max(0.9, 1)), 1) * 10)"}
{'format_valid': True, 'correct': False, 'error_type': 'structure_or_assignment_error', 'value_matches': False, 'missing_variables': [], 'match': None} 

=== A test:44:599:A (LDL Calculated) -> arithmetic_error
).
2. Subtract this result from 64.
3. Subtract the result from 162.

First, 

## Robustness to the wording of the prompts

A score should not depend on how the question is phrased. With `prompt_variants=3`, every prompt is also asked in
two paraphrases ("v2", "v3") that give the same information and ask for the same answer format. The views above use
the default prompts ("v1", whose answers are reused here); `prompt_sensitivity` compares the three wordings.

In [12]:
robust = mk.evaluate_model(model, OUTPUT_DIR, per_calculator=PER_CALCULATOR, batch_size=BATCH_SIZE, prompt_variants=3)
robust

MedCalc-Bench evaluation of Qwen/Qwen2.5-3B-Instruct
  general (end to end): 16.7% of 275 examples (55 calculators)  (prompt variants: 15.6% to 18.2%)
  step by step (165 examples of equation-based calculators):
    E    extraction          86.1%  (prompt variants: 86.1% to 90.3%)
    U    unit conversion     74.2%  (prompt variants: 58.3% to 75.0%)
    K    formula knowledge    3.6%  (prompt variants: 1.2% to 6.7%)
    S    substitution        73.3%  (prompt variants: 73.3% to 89.1%)
    A    arithmetic          61.2%  (prompt variants: 61.2% to 83.0%)
    E2E  end to end          20.6%
  consistency: E2E correct for 80% of the examples whose steps all succeed, 19% when a step fails

In [13]:
robust.prompt_sensitivity.style.format("{:.1%}")

,v1,v2,v3,mean,sd,range
task,,,,,,
E,86.1%,90.3%,87.9%,88.1%,2.1%,4.2%
U,74.2%,58.3%,75.0%,69.2%,9.4%,16.7%
K,3.6%,6.7%,1.2%,3.8%,2.7%,5.5%
S,73.3%,89.1%,84.2%,82.2%,8.1%,15.8%
A,61.2%,63.0%,83.0%,69.1%,12.1%,21.8%
E2E,16.7%,15.6%,18.2%,16.8%,1.3%,2.5%


`consistency` checks the decomposition itself: when every step succeeds, the end-to-end answer should be right; when
a step fails, it should mostly be wrong.

In [14]:
robust.consistency.style.format({"e2e_accuracy": "{:.0%}"})

,examples,e2e_accuracy
case,,
all steps correct,5,80%
a step failed,160,19%


Each item records the name and a hash of its template, to cite the exact prompts in a paper:

In [15]:
robust.items.groupby(["probe", "template"], sort=False).template_hash.first().reset_index().head(6)

,probe,template,template_hash
0,E,v1,51b53cb889
1,U,v1,65c621f8cd
2,K,v1,bdf85aaa07
3,S,v1,b4b759827b
4,A,v1,0a6419395f
5,E2E,v1,f71b90c201


## LLM evaluation, next to the automatic one

An LLM judge grades every answer as well: it reads the task, the registry's reference answer and the probe's
grading rule (for E2E, only the final answer against the ground truth), and says correct or incorrect. The automatic
evaluation stays the reference, so comparing the two measures how far the judge can be trusted, without human labels.

The judge is Qwen3-8B without its thinking mode, deterministic: greedy decoding, one prompt at a time, deterministic
torch kernels; its model revision is recorded with each verdict. The model evaluated is not needed any more: its
answers are saved, so it is unloaded first.

In [16]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

free = [torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())]
judge = HFBackend("Qwen/Qwen3-8B", device=f"cuda:{max(range(len(free)), key=free.__getitem__)}",
                  deterministic=True, chat_template_kwargs={"enable_thinking": False})
t0 = time.time()
judged = mk.evaluate_model(None, OUTPUT_DIR, per_calculator=PER_CALCULATOR, model_id=MODEL_ID, judge=judge)
print(f"{len(judged.llm)} answers graded in {time.time() - t0:.0f} s (judge revision {judge.revision})\n")
judged

1055 answers graded in 1190 s (judge revision b968826d9c46dd6066d109eabc6255188de91218)



MedCalc-Bench evaluation of Qwen/Qwen2.5-3B-Instruct
  general (end to end): 16.7% of 275 examples (55 calculators)
  step by step (165 examples of equation-based calculators):
    E    extraction          86.1%
    U    unit conversion     74.2%
    K    formula knowledge    3.6%
    S    substitution        73.3%
    A    arithmetic          61.2%
    E2E  end to end          20.6%
  consistency: E2E correct for 80% of the examples whose steps all succeed, 19% when a step fails
  LLM evaluation (Qwen/Qwen3-8B): 39.1% correct; agreement with the automatic evaluation 81% (kappa 0.62)

In [17]:
judged.llm_vs_auto.style.format({"auto_accuracy": "{:.1%}", "llm_accuracy": "{:.1%}", "agreement": "{:.0%}",
                                 "kappa": "{:.2f}", "false_acceptance": "{:.0%}", "false_rejection": "{:.0%}"})

,n,unreadable,auto_accuracy,llm_accuracy,agreement,kappa,false_acceptance,false_rejection
task,,,,,,,,
S,165,0,73.3%,83.6%,79%,0.38,59%,7%
E,165,0,86.1%,51.5%,61%,0.19,17%,43%
A,165,0,61.2%,44.8%,67%,0.35,22%,41%
K,165,0,3.6%,6.7%,97%,0.69,3%,0%
E2E,275,0,16.7%,13.8%,95%,0.80,1%,24%
U,120,0,74.2%,55.8%,82%,0.61,0%,25%
all,1055,0,47.9%,39.1%,81%,0.62,9%,29%


`false_acceptance`: share of the answers the automatic scoring rejects that the judge accepts; `false_rejection`: share
of those it accepts that the judge rejects. Disagreements, to read the judge's reasons:

In [18]:
s = judged.scores
disagree = s[s.judge_correct.notna() & (s.judge_correct != s.correct)]
reasons = judged.llm.set_index("item_id").reason
disagree.assign(judge_reason=disagree.item_id.map(reasons))[["item_id", "task", "correct", "judge_correct",
                                                             "judge_reason"]].head(10)

,item_id,task,correct,judge_correct,judge_reason
22,test:11:199:S,S,False,True,The expression correctly substitutes the given values in...
23,test:11:182:S,S,False,True,The expression correctly substitutes the given values in...
24,test:11:191:S,S,False,True,The expression correctly substitutes the given values in...
50,test:57:689:S,S,False,True,The expression correctly substitutes the given values in...
51,test:57:699:S,S,False,True,The expression correctly substitutes the given values in...
52,test:57:682:S,S,False,True,The expression correctly substitutes the given values in...
53,test:57:700:S,S,False,True,The expression correctly substitutes the given values in...
54,test:57:691:S,S,False,True,The expression correctly substitutes the given values in...
62,test:39:540:S,S,True,False,The formula includes the variable 'ag' which should not ...
70,test:7:111:S,S,False,True,The expression correctly substitutes the values into the...


## Saved files

Everything is in `OUTPUT_DIR`: the prompts, the raw generations, and the report as CSV. Rescoring saved generations
needs no model: `mk.evaluate_model(None, OUTPUT_DIR, per_calculator=PER_CALCULATOR, model_id=MODEL_ID)`.

In [19]:
import os
sorted(os.listdir(OUTPUT_DIR))

['by_calculator.csv',
 'by_label.csv',
 'consistency.csv',
 'examples.csv',
 'general.csv',
 'generations.jsonl',
 'items',
 'judge.jsonl',
 'llm_evaluation.csv',
 'llm_vs_auto.csv',
 'prompt_sensitivity.csv',
 'scores.csv',
 'steps.csv']

In [20]:
again = mk.evaluate_model(None, OUTPUT_DIR, per_calculator=PER_CALCULATOR, model_id=MODEL_ID)
print(again.general.loc["all", "accuracy"] == report.general.loc["all", "accuracy"])

True
